# Dusk finite-tail C1 versus CryptoSwap

Reproducible companion to the focused invariant decision report. This notebook reads committed CSV outputs only; it does not fetch network data. CPMM is the control.

In [1]:
from pathlib import Path
import csv, json
ROOT = Path.cwd()
RESULTS = ROOT / 'results'
def rows(name):
    with (RESULTS / name).open(newline='') as source:
        return list(csv.DictReader(source))
summary = json.loads((RESULTS / 'summary.json').read_text())
execution = rows('execution.csv')
jumps = rows('jump_loss.csv')
paths = rows('path_lvr.csv')
restoring = rows('restoring_quotes.csv')
matches = rows('service_constrained_matches.csv')
print('status:', summary['status'])

status: complete


In [2]:
print('Parameter identity:', summary['parameter_identity']['equation'])
print('Safety cells:', summary['safety']['tested_parameter_cells'])
print('Off-center solver failures:', summary['safety']['offcenter_solver_failures'])
print('Global quote ordering claimed:', summary['safety']['global_quote_order_claimed'])

Parameter identity: peak_depth = 2 * Curve A; fade_scale = Curve gamma
Safety cells: 420
Off-center solver failures: 0
Global quote ordering claimed: False


In [3]:
balanced = summary['decision_examples']['balanced']
print('Balanced profile — execution premium (bps)')
for size, values in balanced['execution_premium_bps_by_trade_size'].items():
    print(size, {k: round(v, 4) for k, v in values.items()})
print('2x one-jump loss:', {k: round(v, 3) for k, v in balanced['one_jump_2x_loss_bps_initial_tvl'].items()})
print('CryptoSwap excess loss removed:', round(100 * balanced['fraction_of_cryptoswap_excess_loss_removed_by_dusk'], 2), '%')

Balanced profile — execution premium (bps)
10 {'cpmm': 20.0, 'cryptoswap': 0.0997, 'dusk_c1': 0.0997}
100 {'cpmm': 200.0, 'cryptoswap': 1.1591, 'dusk_c1': 1.1591}
1000 {'cpmm': 2000.0, 'cryptoswap': 412.5182, 'dusk_c1': 1707.3171}
2000 {'cpmm': 4000.0, 'cryptoswap': 2044.87, 'dusk_c1': 3827.1605}
2x one-jump loss: {'cpmm': 857.864, 'cryptoswap': 1351.8, 'dusk_c1': 893.264}
CryptoSwap excess loss removed: 92.83 %


In [4]:
print('Service-constrained CryptoSwap matches')
for row in matches:
    print(row['dusk_profile'], 'Dusk loss reduction (bps):', round(float(row['dusk_loss_reduction_vs_no_worse_execution_curve_bps']), 3))

Service-constrained CryptoSwap matches
broad Dusk loss reduction (bps): 187.306
balanced Dusk loss reduction (bps): 91.509
aggressive Dusk loss reduction (bps): 251.683


In [5]:
print('Balanced profile — cumulative path LVR (bps initial TVL)')
for scenario, values in summary['path_lvr_examples']['balanced'].items():
    print(scenario, {k: round(v, 3) for k, v in values.items()})

Balanced profile — cumulative path LVR (bps initial TVL)
alternating_chop_10pct {'cpmm': 885.719, 'cryptoswap': 3009.413, 'dusk_c1': 1346.159}
jump_reversal_2x {'cpmm': 1464.466, 'cryptoswap': 1887.093, 'dusk_c1': 1473.316}
single_jump_2x {'cpmm': 857.864, 'cryptoswap': 1351.8, 'dusk_c1': 893.264}
staircase_2x {'cpmm': 36.202, 'cryptoswap': 48.538, 'dusk_c1': 40.035}
staircase_4x {'cpmm': 87.398, 'cryptoswap': 94.641, 'dusk_c1': 91.104}
volatility_cluster {'cpmm': 111.893, 'cryptoswap': 489.157, 'dusk_c1': 229.263}


In [6]:
print('Full restore from a native 2x state — output difference vs same-reserve CPMM (bps)')
for profile, values in summary['decision_examples'].items():
    print(profile, {k: round(v, 3) for k, v in values['full_restore_from_2x_output_difference_vs_same_reserve_cpmm_bps'].items()})

Full restore from a native 2x state — output difference vs same-reserve CPMM (bps)
aggressive {'cpmm': 0.0, 'cryptoswap': -774.636, 'dusk_c1': -12.067}
balanced {'cpmm': 0.0, 'cryptoswap': -1352.779, 'dusk_c1': -120.287}
broad {'cpmm': 0.0, 'cryptoswap': -1974.058, 'dusk_c1': -1108.084}


In [7]:
assert summary['safety']['all_passed']
assert summary['safety']['offcenter_solver_failures'] == 0
assert all(float(row['dusk_loss_reduction_vs_no_worse_execution_curve_bps']) > 0 for row in matches)
assert summary['solver_proxy']['actual_compute_units'] is None
print('Notebook reconciliation: PASS')

Notebook reconciliation: PASS
